# ODS v2: Premium finance loans

Builds 1 ods tables in `ods_v2`: `PolicyLoanStatus`.

Needs these notebooks to have succeeded first: 10_Policy.

Maintained by hand: first generated by `CGP_analysis`, no longer regenerated.

**If a cell fails**

1. Read the description above the failing cell: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first four cells again first (lakehouse, parameters, setup, start log).


In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "ChillLH"
    }
}

In [ ]:
%%pyspark
# Parameters (set by the 01 - ODS Build v2 pipeline; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("11_Policy_Loan", "started")


## PolicyLoanStatus


**`${ods.stg}.odsv2_loan_day`**

One row per loan and load day (TechInsertDate), the live row first, then the latest id.

- **Reads:** dlk.cda_loanse

In [ ]:
create or replace table ${ods.stg}.odsv2_loan_day as
select  LoanReference,
        cast(TechInsertDate as date)                                    as SnapshotDate,
        LoanStatusId,
        nullif(upper(trim(PolicyCode)), '')                                                as PolicyCode,
        nullif(upper(trim(ClientCode)), '')                                                as ClientCode,
        BranchId,
        Premium,
        CreditCharge,
        ReceivedDate,
        InceptionDate,
        PolicyStatusId,
        LoanMappingStatusId,
        LoanStatusForUnauthorisedId,
        LoanStatusForCancellationId
from    (
        select  *, row_number() over (partition by LoanReference, cast(TechInsertDate as date)
                                      order by case when TechDeleteDate is null then 0 else 1 end, Id desc) as rn
        from    ${ods.dlk}.cda_loanse
        where   LoanReference is not null and TechInsertDate is not null
        ) l
where   rn = 1

**`${ods.stg}.odsv2_loan_run`**

Each load day numbered into runs of the same LoanStatusId per loan (gaps and islands).

- **Reads:** stg.odsv2_loan_day

In [ ]:
create or replace table ${ods.stg}.odsv2_loan_run as
select  d.*,
        sum(d.IsChange) over (partition by d.LoanReference order by d.SnapshotDate
                              rows between unbounded preceding and current row) as RunNo
from    (
        select  l.*,
                case when lag(l.LoanStatusId) over (partition by l.LoanReference order by l.SnapshotDate)
                          <=> l.LoanStatusId then 0 else 1 end          as IsChange
        from    ${ods.stg}.odsv2_loan_day l
        ) d

**`{target}.PolicyLoanStatus`**

Close Brothers premium-finance loans (the only loan provider in the data; Premium Credit is not loaded) as a status

- **history:** the daily loan file (one row per loan per load day, about 25M rows) collapsed to one row per run of the same LoanStatusId, with the first and last load day seen and the next run's first day. LoanStatusId has no code table; read from its transitions (D-E1.9, for client validation) as 2 pending / new, 6 active, 5 in arrears, 4 default, 3 cancelled / terminated (7: 24 rows, unknown). Superseded reloads (TechDeleteDate set) lose to the live row of the same day. The client's name is not taken.
- **Grain:** one row per loan and status run (consecutive daily snapshots with the same LoanStatusId)
- **Business key:** LoanReference + StatusFromDate
- **History:** versioned (StatusFromDate / StatusToDate; daily snapshots collapsed to status changes)
- **Cut-off:** none
- **Source:** dlk.cda_loanse
- **Reads:** stg.odsv2_loan_run, ods_v2.Policy

In [ ]:
create or replace table ${ods.target}.PolicyLoanStatus as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  r.LoanReference,
                p.PolicyId,
                r.PolicyCode,
                r.ClientCode,
                r.LoanStatusId,
                lag(r.LoanStatusId) over (partition by r.LoanReference order by r.StatusFromDate) as PreviousLoanStatusId,
                r.StatusFromDate,
                r.LastSeenDate,
                lead(r.StatusFromDate) over (partition by r.LoanReference order by r.StatusFromDate) as StatusToDate,
                r.SnapshotDays,
                r.BranchId,
                r.Premium,
                r.CreditCharge,
                r.ReceivedDate,
                r.InceptionDate,
                r.PolicyStatusId,
                r.LoanMappingStatusId,
                r.LoanStatusForUnauthorisedId,
                r.LoanStatusForCancellationId
        from    (
                select  LoanReference, RunNo, min(LoanStatusId) as LoanStatusId,
                        min(SnapshotDate)                                       as StatusFromDate,
                        max(SnapshotDate)                                       as LastSeenDate,
                        count(*)                                                as SnapshotDays,
                        min_by(PolicyCode, SnapshotDate)                        as PolicyCode,
                        min_by(ClientCode, SnapshotDate)                        as ClientCode,
                        min_by(BranchId, SnapshotDate)                          as BranchId,
                        min_by(Premium, SnapshotDate)                           as Premium,
                        min_by(CreditCharge, SnapshotDate)                      as CreditCharge,
                        min_by(ReceivedDate, SnapshotDate)                      as ReceivedDate,
                        min_by(InceptionDate, SnapshotDate)                     as InceptionDate,
                        min_by(PolicyStatusId, SnapshotDate)                    as PolicyStatusId,
                        min_by(LoanMappingStatusId, SnapshotDate)               as LoanMappingStatusId,
                        min_by(LoanStatusForUnauthorisedId, SnapshotDate)       as LoanStatusForUnauthorisedId,
                        min_by(LoanStatusForCancellationId, SnapshotDate)       as LoanStatusForCancellationId
                from    ${ods.stg}.odsv2_loan_run
                group by LoanReference, RunNo
                ) r
        left join ${ods.target}.Policy p    on p.PolicyCode = r.PolicyCode
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "PolicyLoanStatus", "bk": ["LoanReference", "StatusFromDate"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("11_Policy_Loan", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("11_Policy_Loan", "succeeded")
